# 🏆 飞书文档 API 终极全生命周期验证 (Lark API Absolute Showcase)

> **作者**: MetaBlog Agent Sandbox
> **说明**: 展示从创建到极其丰富复杂的文档块（包含序列表、代码块、数学公式、大表格、定点增删改等）的 100% 稳健生命周期。
> **核心准则**: 每一项原子操作后必进行状态读取，确保所有底层结构严格按照预期展开。

In [1]:
import os
import time
import json
from dotenv import load_dotenv
from pathlib import Path
from feishu_client import FeishuClient
import requests

# 0. 初始化探测
load_dotenv(Path('../../.env'))
client = FeishuClient()
token = client.get_tenant_access_token()
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

def verify_step(did, action_name):
    """辅助：在每个步骤后读取并验证文档状态"""
    print(f"\n--- [验证] {action_name} ---")
    res = client.api('GET', f'/docx/v1/documents/{did}/raw_content')
    blocks = client.api('GET', f'/docx/v1/documents/{did}/blocks/{did}/children')
    items = blocks.get('items', [])
    content = res.get('content', '').replace('\n', ' ')
    print(f"  [RESULT] 块树追踪仪: 获取到当前 {len(items)} 个顶层块。")
    print(f"  [TXT快照] {content[:150]}...")

print('✅ 验证环境准备就绪，即将进入全栈沉浸式文档重构!')

✅ 验证环境准备就绪，即将进入全栈沉浸式文档重构!


## 0. 身份发现 (Identity Discovery)

In [2]:
# 只要你有用户的邮箱、手机号或用户 ID，就能换取其在飞书宇宙中的唯一 OpenID
# 请在下面输入你自己的信息进行测试
TEST_EMAIL = "" 
TEST_MOBILE = "13586820267" 
TEST_USER_ID = "811ed377" # <--- 填入你截图里的用户 ID！

payload = {}
if TEST_EMAIL: payload['emails'] = [TEST_EMAIL]
if TEST_MOBILE: payload['mobiles'] = [TEST_MOBILE]
if TEST_USER_ID: payload['user_ids'] = [TEST_USER_ID]

try:
    if not payload:
        print("ℹ️ 请在上面填入信息以测试身份锁定。")
        MY_OPEN_ID = None
    else:
        print(f"--- [DEBUGLOG] 正在发起身份锁定请求... ---")
        ident_res = client.api('POST', '/contact/v3/users/batch_get_id', 
                              json_data=payload,
                              params={'user_id_type': 'open_id'})
        
        # 刚才测试代码里的“救命”日志：
        print("--- [DEBUGLOG] 飞书原始返回 JSON ---")
        print(json.dumps(ident_res, indent=2, ensure_ascii=False))
        
        user_list = ident_res.get('user_list', [])
        if user_list:
            # 兼容性读取：优先看 user_id (因为飞书在返回 open_id 时字段常命名为 user_id)
            first_user = user_list[0]
            MY_OPEN_ID = first_user.get('user_id') or first_user.get('open_id')
            
            if MY_OPEN_ID:
                print(f"\n✅ [成功] 锁定 ID: {MY_OPEN_ID}")
            else:
                print("\n❌ [失败] 返回了用户但未包含有效 ID 字段")
        else:
            MY_OPEN_ID = None
            print("\n❌ 未能在当前租户下找到对应用户，请确认信息准确或应用权限。")
except Exception as e:
    MY_OPEN_ID = None
    print(f"\n⚠️ 身份查询失败: {e}")


--- [DEBUGLOG] 正在发起身份锁定请求... ---
--- [DEBUGLOG] 飞书原始返回 JSON ---
{
  "user_list": [
    {
      "mobile": "13586820267",
      "user_id": "ou_5fd95247243d26b82a1e1a5fdefe973d"
    }
  ]
}

✅ [成功] 锁定 ID: ou_5fd95247243d26b82a1e1a5fdefe973d


## 1. 建立空域，规避同步重命名延迟

In [3]:
# 文档创立期就敲定名字！并追加休眠机制保证 Drive API 的索引落盘！
r_doc = client.api('POST', '/docx/v1/documents', json_data={'title': 'MetaBlog: The Ultimate API Guide'})
doc_id = r_doc['document']['document_id']
print(f"✅ 文档建立于 {doc_id}，正在沉睡 2.5 秒等待飞书数据层跨区同步...")
time.sleep(2.5)

# [更新] 权限自动下放逻辑：如果刚才拿到了 OpenID，就把这个文档的管理权分享给该用户
if 'MY_OPEN_ID' in locals() and MY_OPEN_ID:
    try:
        perm_res = client.api('POST', f'/drive/v1/permissions/{doc_id}/members',
                             params={'type': 'docx'}, 
                             json_data={
                                 'member_type': 'openid',
                                 'member_id': MY_OPEN_ID,
                                 'perm': 'full_access' # 赋予最高管理权限
                             })
        print(f"✅ 权限已成功下放给用户 [{MY_OPEN_ID}] (Full Access)")
    except Exception as e:
        print(f"⚠️ 权限下放失败: {e}")

verify_step(doc_id, "第一阶段：文档创建与权限下放完成")

✅ 文档建立于 UTZpd7j9ToaVtYx7R9lcB7OKnle，正在沉睡 2.5 秒等待飞书数据层跨区同步...
✅ 权限已成功下放给用户 [ou_5fd95247243d26b82a1e1a5fdefe973d] (Full Access)

--- [验证] 第一阶段：文档创建与权限下放完成 ---
  [RESULT] 块树追踪仪: 获取到当前 0 个顶层块。
  [TXT快照] MetaBlog: The Ultimate API Guide ...


## 2. 基础与多样性排版流 (Rich Text & Lists)

In [4]:
# 我们要一口气压入大量的排版：一二级标题、有序、无序、引用、基础文本
rich_blocks = [
    {'block_type': 3, 'heading1': {'elements': [{'text_run': {'content': '核心结构图鉴'}}]}},
    {'block_type': 4, 'heading2': {'elements': [{'text_run': {'content': 'A. 多样化列表呈现'}}]}},
    {'block_type': 2, 'text': {'elements': [{'text_run': {'content': '飞书不仅支持字幅写入，也具备原生列表转换层：'}}]}},
    {'block_type': 13, 'ordered': {'elements': [{'text_run': {'content': '首先，这是一条坚定的有序列表点1'}}]}},
    {'block_type': 13, 'ordered': {'elements': [{'text_run': {'content': '接着，这是它的兄弟节点有序列表2'}}]}},
    {'block_type': 12, 'bullet': {'elements': [{'text_run': {'content': '当然，也有随需应变的无序圆点A'}}]}},
    {'block_type': 12, 'bullet': {'elements': [{'text_run': {'content': '紧接着是配套的无序圆点B'}}]}},
    {'block_type': 15, 'quote': {'elements': [{'text_run': {'content': '“凡事追求极致的确定性，方能突破模型的局限边界” —— 邵承源'}}]}},
    {'block_type': 22, 'divider': {}},  # 华丽的分割线
    {'block_type': 4, 'heading2': {'elements': [{'text_run': {'content': 'B. CRUD 定位靶点区'}}]}},
    {'block_type': 2, 'text': {'elements': [{'text_run': {'content': '【待修改】本段会被 PATCH 指令高亮！'}}]}},
    {'block_type': 2, 'text': {'elements': [{'text_run': {'content': '【待删除】这是被献祭的一行垃圾字符数据。'}}]}}
]
client.api('POST', f'/docx/v1/documents/{doc_id}/blocks/{doc_id}/children', json_data={'children': rich_blocks})
verify_step(doc_id, "排版矩阵空投成功")


--- [验证] 排版矩阵空投成功 ---
  [RESULT] 块树追踪仪: 获取到当前 12 个顶层块。
  [TXT快照] MetaBlog: The Ultimate API Guide 核心结构图鉴 A. 多样化列表呈现 飞书不仅支持字幅写入，也具备原生列表转换层： 首先，这是一条坚定的有序列表点1 接着，这是它的兄弟节点有序列表2 当然，也有随需应变的无序圆点A 紧接着是配套的无序圆点B “凡事追求极致的确定性，方...


## 3. 定点猎杀 (Delete from Tail)

In [5]:
# 上一步我们塞入了 12 个块，所以当前的块总数变为 12 (0到11)。
# 我们打算删除最后一个文字块（也就是我们的垃圾字符数据，在原数组的索引11处）。
# 因为它是最后追加的，为了稳定我们重新先拉取获取它的真实 index
lists = client.api('GET', f'/docx/v1/documents/{doc_id}/blocks/{doc_id}/children')
items = lists.get('items', [])

# 寻找含有"待删除"文字的 Block
death_index = next((i for i, b in enumerate(items) if "待删除" in json.dumps(b, ensure_ascii=False)), None)

if death_index is not None:
    client.api('DELETE', f'/docx/v1/documents/{doc_id}/blocks/{doc_id}/children/batch_delete', 
               json_data={'start_index': death_index, 'end_index': death_index + 1})
    print(f"✅ 从数组下标 [{death_index}] 精确消灭了靶点 Block！")
else:
    print("❌ 未能找到待删除靶点！")

verify_step(doc_id, "精确制导删除")

✅ 从数组下标 [11] 精确消灭了靶点 Block！

--- [验证] 精确制导删除 ---
  [RESULT] 块树追踪仪: 获取到当前 11 个顶层块。
  [TXT快照] MetaBlog: The Ultimate API Guide 核心结构图鉴 A. 多样化列表呈现 飞书不仅支持字幅写入，也具备原生列表转换层： 首先，这是一条坚定的有序列表点1 接着，这是它的兄弟节点有序列表2 当然，也有随需应变的无序圆点A 紧接着是配套的无序圆点B “凡事追求极致的确定性，方...


## 4. 极致覆写 (Precision PATCH)

In [6]:
# 重新获取，定位【待修改】靶点
lists = client.api('GET', f'/docx/v1/documents/{doc_id}/blocks/{doc_id}/children')
target_id = next((b['block_id'] for b in lists.get('items', []) if "待修改" in json.dumps(b, ensure_ascii=False)), None)

if target_id:
    # 发动携带 Style (斜体和下划线) 的 PATCH 猛攻
    requests.patch(f'https://open.feishu.cn/open-apis/docx/v1/documents/{doc_id}/blocks/{target_id}',
                   headers=headers, 
                   json={'update_text_elements': {'elements': [{'text_run': {'content': '【我已重生】：不再是原先粗陋的待修改靶标了！', 'text_element_style': {'bold': True, 'italic': True, 'underline': True}}}]}})
    print(f"✅ 成功定点升级 Block: {target_id}")
else:
    print("❌ 未能找到待修改靶点！")
    
verify_step(doc_id, "样式叠加与强制覆写")

✅ 成功定点升级 Block: doxcn9ic5U6NmaGzrIM1SV1rW8c

--- [验证] 样式叠加与强制覆写 ---
  [RESULT] 块树追踪仪: 获取到当前 11 个顶层块。
  [TXT快照] MetaBlog: The Ultimate API Guide 核心结构图鉴 A. 多样化列表呈现 飞书不仅支持字幅写入，也具备原生列表转换层： 首先，这是一条坚定的有序列表点1 接着，这是它的兄弟节点有序列表2 当然，也有随需应变的无序圆点A 紧接着是配套的无序圆点B “凡事追求极致的确定性，方...


## 5. 高维极客块：源码片段与公式推导 (Code & Equations [Type 16])

In [7]:
# 注意公式属于块类型 16! (飞书文档深埋的暗坑)
geek_blocks = [
    {'block_type': 4, 'heading2': {'elements': [{'text_run': {'content': 'C. 极客硬核推演'}}]}},
    {'block_type': 14, 'code': {'style': {'language': 2}, 'elements': [{'text_run': {'content': 'def metablog_agent_run():\n    print("Agent is fully operational!")\n    return 0'}}]}},
    {'block_type': 2, 'text': {'elements': [{'text_run': {'content': '定理公式推导：'}}, {'equation': {'content': 'P(A|B) = \\frac{P(B|A)P(A)}{P(B)}'}}]}}
]
client.api('POST', f'/docx/v1/documents/{doc_id}/blocks/{doc_id}/children', json_data={'children': geek_blocks})
verify_step(doc_id, "贝叶斯与程序的史诗交响")


--- [验证] 贝叶斯与程序的史诗交响 ---
  [RESULT] 块树追踪仪: 获取到当前 14 个顶层块。
  [TXT快照] MetaBlog: The Ultimate API Guide 核心结构图鉴 A. 多样化列表呈现 飞书不仅支持字幅写入，也具备原生列表转换层： 首先，这是一条坚定的有序列表点1 接着，这是它的兄弟节点有序列表2 当然，也有随需应变的无序圆点A 紧接着是配套的无序圆点B “凡事追求极致的确定性，方...


## 6. 表格矩阵的全并发数据穿透与 3QPS 保护

In [8]:
print("--- 构建 5x5 的数据要塞 ---")
table_req = {'block_type': 31, 'table': {'property': {'column_size': 5, 'row_size': 5}}}
r_table = client.api('POST', f'/docx/v1/documents/{doc_id}/blocks/{doc_id}/children', json_data={'children': [table_req]})
cell_ids = r_table['children'][0]['table']['cells']

for idx, cid in enumerate(cell_ids):
    if idx > 0 and idx % 3 == 0: time.sleep(0.4) 
    
    r_cell = client.request('GET', f'/docx/v1/documents/{doc_id}/blocks/{cid}')
    anchor_id = r_cell["data"]["block"]["children"][0]
    # 无副作用定点填充
    requests.patch(f'https://open.feishu.cn/open-apis/docx/v1/documents/{doc_id}/blocks/{anchor_id}',
                   headers=headers, 
                   json={'update_text_elements': {'elements': [{'text_run': {'content': f'({idx//5},{idx%5})'}}]}})

print("✅ 25格矩阵数据轰炸不掉线！")
verify_step(doc_id, "表格压力过载平衡测试")

--- 构建 5x5 的数据要塞 ---
✅ 25格矩阵数据轰炸不掉线！

--- [验证] 表格压力过载平衡测试 ---
  [RESULT] 块树追踪仪: 获取到当前 15 个顶层块。
  [TXT快照] MetaBlog: The Ultimate API Guide 核心结构图鉴 A. 多样化列表呈现 飞书不仅支持字幅写入，也具备原生列表转换层： 首先，这是一条坚定的有序列表点1 接着，这是它的兄弟节点有序列表2 当然，也有随需应变的无序圆点A 紧接着是配套的无序圆点B “凡事追求极致的确定性，方...


## 7. 见证奇迹 (Final Rendering Report)

In [9]:
print(f"\n{'='*60}\n🏆 【终局呈现】\nURL: https://feishu.cn/docx/{doc_id}\n{'='*60}")
final_res = client.api('GET', f'/docx/v1/documents/{doc_id}/raw_content')
print(final_res.get('content'))
print("=" * 60)


🏆 【终局呈现】
URL: https://feishu.cn/docx/UTZpd7j9ToaVtYx7R9lcB7OKnle
MetaBlog: The Ultimate API Guide
核心结构图鉴
A. 多样化列表呈现
飞书不仅支持字幅写入，也具备原生列表转换层：
首先，这是一条坚定的有序列表点1
接着，这是它的兄弟节点有序列表2
当然，也有随需应变的无序圆点A
紧接着是配套的无序圆点B
“凡事追求极致的确定性，方能突破模型的局限边界” —— 邵承源

B. CRUD 定位靶点区
【我已重生】：不再是原先粗陋的待修改靶标了！
C. 极客硬核推演
def metablog_agent_run():
    print("Agent is fully operational!")
    return 0
定理公式推导： 


(0,0)

(0,1)

(0,2)

(0,3)

(0,4)

(1,0)

(1,1)

(1,2)

(1,3)

(1,4)

(2,0)

(2,1)

(2,2)

(2,3)

(2,4)

(3,0)

(3,1)

(3,2)

(3,3)

(3,4)

(4,0)

(4,1)

(4,2)

(4,3)

(4,4)

